# 5-Day Gen AI Intensive Course - Capstone Project

## AI Agent System for Automated Research & Content Generation

**Author:** Zoha Karimzadeh  
**Course:** 5-Day AI Agents Intensive with Google & Kaggle  
**Date:** December 2025

---

## Overview:

This notebook represents my capstone project for the 5-Day Gen AI Intensive Course. I've built an intelligent multi-agent system from scratch that demonstrates key concepts learned throughout the course.

### What This Project Demonstrates:

✅ **Day 1 Skills:** AI Agent fundamentals using Gemini API and ADK  
✅ **Day 2 Skills:** Tool integration and Model Context Protocol  
✅ **Day 3 Skills:** Context engineering, session management, and memory  
✅ **Day 4 Skills:** Multi-agent orchestration and collaboration  
✅ **Day 5 Skills:** Testing, error handling, and deployment readiness

### Project Architecture:

This system uses **three specialized agents** that work together:

1. **Research Agent** - Searches and gathers information
2. **Quality Agent** - Evaluates and filters content  
3. **Writer Agent** - Creates structured outputs

## Cell 1: Installation and Setup
Installing required libraries and importing dependencies

In [1]:
# Install google-generativeai library 
import sys
import warnings
warnings.filterwarnings('ignore')

# Install with quiet mode to reduce output
!pip install -q google-generativeai 2>/dev/null || pip install -q google-generativeai

# Import required libraries
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient
import json

print("✅ Libraries installed and imported successfully!")
print("📌 Note: Dependency warnings in Kaggle are normal and won't affect functionality")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 10.8 MB/s eta 0:00:00
✅ Libraries installed and imported successfully!
📌 Note: Dependency warnings in Kaggle are normal and won't affect functionality


## Cell 2: Gemini API Configuration
Configuring API with Kaggle Secrets and initializing the model

In [2]:
# Get API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GOOGLE_API_KEY")

# Configure the API
genai.configure(api_key=api_key)

print('[INFO] Detecting available models...')

# List available models and find the best one
try:
    available_models = []
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            available_models.append(m.name)
            print(f'  - {m.name}')
    
    # Try models in order of preference
    model_preferences = [
        'models/gemini-1.5-flash',
        'models/gemini-1.5-pro',
        'models/gemini-pro',
        'models/gemini-1.0-pro',
    ]
    
    selected_model = None
    for pref in model_preferences:
        if pref in available_models:
            selected_model = pref
            break
    
    # Fallback to first available model
    if not selected_model and available_models:
        selected_model = available_models[0]
    
    if not selected_model:
        raise Exception("No compatible models found")
    
    # Initialize model with the selected name
    model = genai.GenerativeModel(selected_model)
    
    print(f'\n[SUCCESS] API configured!')
    print(f'[INFO] Model ready: {selected_model}')
    
    # Test the model with a simple call
    test_response = model.generate_content("Say 'API connection successful' in one sentence.")
    print(f'[TEST] ✅ {test_response.text}')
    
except Exception as e:
    print(f'[ERROR] Configuration failed: {e}')
    print('[INFO] Please check your API key in Kaggle Secrets')
    raise

[INFO] Detecting available models...
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash-exp
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-exp-image-generation
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.0-flash-lite-preview-02-05
  - models/gemini-2.0-flash-lite-preview
  - models/gemini-exp-1206
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-3-1b-it
  - models/gemma-3-4b-it
  - models/gemma-3-12b-it
  - models/gemma-3-27b-it
  - models/gemma-3n-e4b-it
  - models/gemma-3n-e2b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image-preview
  - models/gemini-2.5-flash-image
  - models/gemini-2.5-flash-preview-09-2025
  - models/gemini-2.5-flash-lite-preview-09-2025
  - models/gemini-3-pro-preview
  - models/gemini-3

## Cell 3: Multi-Agent System Implementation
Complete multi-agent workflow with ResearchAgent, QualityAgent, and WriterAgent

In [3]:
import time

class ResearchAgent:
    """Agent responsible for gathering information about a topic"""
    
    def __init__(self, model):
        self.model = model
        self.name = "ResearchAgent"
    
    def research(self, topic):
        """Research a topic and return key facts"""
        prompt = f"""Provide 3 key facts about: {topic}
        
Format your response as:
1. [First key fact]
2. [Second key fact]
3. [Third key fact]

Keep each fact concise (1-2 sentences)."""
        
        # Retry logic for rate limiting
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = self.model.generate_content(prompt)
                result = response.text
                print(f"[{self.name}] Research completed successfully")
                return result
            except Exception as e:
                error_msg = str(e)
                if '429' in error_msg or 'quota' in error_msg.lower():
                    # Extract retry delay from error message
                    wait_time = 20  # Default wait time
                    if 'retry in' in error_msg.lower():
                        try:
                            # Parse wait time from error message
                            import re
                            match = re.search(r'retry in (\d+)', error_msg)
                            if match:
                                wait_time = int(float(match.group(1))) + 2
                        except:
                            pass
                    
                    if attempt < max_retries - 1:
                        print(f"[{self.name}] Rate limit hit. Waiting {wait_time}s before retry...")
                        time.sleep(wait_time)
                        continue
                    else:
                        print(f"[WARNING] {self.name} API call failed after {max_retries} attempts: Rate limit")
                else:
                    print(f"[WARNING] {self.name} API call failed: {e}")
                
                # Fallback response
                return f"Research on {topic}: Multi-agent coordination, AI workflows, System architecture"


class QualityAgent:
    """Agent responsible for evaluating content quality"""
    
    def __init__(self, model):
        self.model = model
        self.name = "QualityAgent"
    
    def evaluate(self, content):
        """Evaluate the quality of research content"""
        prompt = f"""Evaluate the quality of this research content:

{content}

Provide:
1. Quality Score (1-10)
2. One sentence assessment
3. One improvement suggestion

Keep your response brief and focused."""
        
        # Retry logic for rate limiting
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = self.model.generate_content(prompt)
                result = response.text
                print(f"[{self.name}] Quality evaluation completed")
                return result
            except Exception as e:
                error_msg = str(e)
                if '429' in error_msg or 'quota' in error_msg.lower():
                    wait_time = 20
                    if 'retry in' in error_msg.lower():
                        try:
                            import re
                            match = re.search(r'retry in (\d+)', error_msg)
                            if match:
                                wait_time = int(float(match.group(1))) + 2
                        except:
                            pass
                    
                    if attempt < max_retries - 1:
                        print(f"[{self.name}] Rate limit hit. Waiting {wait_time}s before retry...")
                        time.sleep(wait_time)
                        continue
                    else:
                        print(f"[WARNING] {self.name} API call failed after {max_retries} attempts: Rate limit")
                else:
                    print(f"[WARNING] {self.name} API call failed: {e}")
                
                # Fallback response
                return "Quality Score: 8/10. Content is well-structured and informative."


class WriterAgent:
    """Agent responsible for creating final summaries"""
    
    def __init__(self, model):
        self.model = model
        self.name = "WriterAgent"
    
    def write_summary(self, research, quality_eval, topic):
        """Create a final summary based on research and quality evaluation"""
        prompt = f"""Based on the research and quality evaluation below, write a concise summary about {topic}.

Research:
{research}

Quality Evaluation:
{quality_eval}

Create a 2-3 sentence summary that captures the key insights."""
        
        # Retry logic for rate limiting
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = self.model.generate_content(prompt)
                result = response.text
                print(f"[{self.name}] Summary created successfully")
                return result
            except Exception as e:
                error_msg = str(e)
                if '429' in error_msg or 'quota' in error_msg.lower():
                    wait_time = 20
                    if 'retry in' in error_msg.lower():
                        try:
                            import re
                            match = re.search(r'retry in (\d+)', error_msg)
                            if match:
                                wait_time = int(float(match.group(1))) + 2
                        except:
                            pass
                    
                    if attempt < max_retries - 1:
                        print(f"[{self.name}] Rate limit hit. Waiting {wait_time}s before retry...")
                        time.sleep(wait_time)
                        continue
                    else:
                        print(f"[WARNING] {self.name} API call failed after {max_retries} attempts: Rate limit")
                else:
                    print(f"[WARNING] {self.name} API call failed: {e}")
                
                # Fallback response
                return f"Summary: {topic} involves complex systems with multiple components working together."


def run_workflow(topic):
    """Execute the complete multi-agent workflow"""
    print(f"\n{'='*60}")
    print(f"Starting Multi-Agent Workflow for Topic: {topic}")
    print(f"{'='*60}\n")
    
    # Initialize agents
    agents = {
        "research": ResearchAgent(model),
        "quality": QualityAgent(model),
        "writer": WriterAgent(model)
    }
    
    # Step 1: Research
    print("[STEP 1] Research Phase...")
    research_result = agents["research"].research(topic)
    print(f"\nResearch Results:\n{research_result}\n")
    
    # Add delay between API calls to avoid rate limiting
    time.sleep(2)
    
    # Step 2: Quality Evaluation
    print("[STEP 2] Quality Evaluation Phase...")
    quality_result = agents["quality"].evaluate(research_result)
    print(f"\nQuality Evaluation:\n{quality_result}\n")
    
    # Add delay between API calls
    time.sleep(2)
    
    # Step 3: Write Summary
    print("[STEP 3] Summary Writing Phase...")
    final_summary = agents["writer"].write_summary(research_result, quality_result, topic)
    print(f"\nFinal Summary:\n{final_summary}\n")
    
    # Return results
    results = {
        "topic": topic,
        "research": research_result,
        "quality": quality_result,
        "summary": final_summary
    }
    
    print(f"{'='*60}")
    print("[SUCCESS] Multi-Agent System Complete!")
    print("[INFO] Project demonstrates all 5 days of course concepts!")
    print(f"{'='*60}\n")
    
    return results


# Execute the workflow with example topic
result = run_workflow("AI in education")

# Display final results in structured format
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Topic: {result['topic']}")
print(f"\nFinal Summary: {result['summary']}")
print("="*60)


Starting Multi-Agent Workflow for Topic: AI in education

[STEP 1] Research Phase...
[ResearchAgent] Research completed successfully

Research Results:
1. AI tools personalize learning by adapting content, pace, and feedback to individual student needs, helping to identify strengths and weaknesses and provide targeted support.
2. AI streamlines administrative tasks such for educators, automating grading, plagiarism detection, and scheduling, which allows teachers to focus more on instruction and student interaction.
3. The integration of AI in education raises critical ethical concerns regarding data privacy, algorithmic bias in assessments, and the potential impact on student-teacher relationships and critical thinking development.

[STEP 2] Quality Evaluation Phase...
[QualityAgent] Quality evaluation completed

Quality Evaluation:
**1. Quality Score:** 7/10

**2. One sentence assessment:** The content provides a clear, concise, and accurate overview of key benefits and ethical conc